## Observed incidence by predicted risk

Compare SOC with SOC+NMR using the fold-specific absolute risks in `outputs/main/absolute_risks/`. People are ranked by `pred_risk_5yr`. The percentile figure shows observed incidence across all risk percentiles. The dumbbell figures show observed 5-year incidence in the top 1%, 5%, and 10%. COPD is saved separately.


In [ ]:
library(tidyverse)
library(here)

In [ ]:
source(here("helpers/oof_enrichment.r"))

In [ ]:
risk_dir <- here("outputs/main/absolute_risks")
fig_dir <- here("outputs/figures")
t_star <- 5
risk_col <- sprintf("pred_risk_%dyr", t_star)
n_percentiles <- 100
top_props <- c(0.01, 0.05, 0.10)

disease_specs <- list(
  t2d = list(
    title = "Type 2 Diabetes", short = "T2D", y_max = 0.80,
    soc_file = "t2d_a1c_marker_fold_baseline_risk.rds",
    nmr_file = "t2d_a1c_marker_nmr_fold_baseline_risk.rds"
  ),
  ckd = list(
    title = "Chronic Kidney Disease", short = "CKD", y_max = 0.99,
    soc_file = "ckd_egfr_marker_fold_baseline_risk.rds",
    nmr_file = "ckd_egfr_marker_nmr_fold_baseline_risk.rds"
  ),
  nafld = list(
    title = "MASLD (Liver Disease)", short = "MASLD", y_max = 0.35,
    soc_file = "nafld_fib4_marker_fold_baseline_risk.rds",
    nmr_file = "nafld_fib4_marker_nmr_fold_baseline_risk.rds"
  ),
  hf_cm = list(
    title = "Heart Failure", short = "Heart Failure", y_max = 0.40,
    soc_file = "hf_cm_prevent_ingredients_marker_fold_baseline_risk.rds",
    nmr_file = "hf_cm_prevent_ingredients_marker_nmr_fold_baseline_risk.rds"
  ),
  acute_mi = list(
    title = "Acute MI", short = "Acute MI", y_max = 0.16,
    soc_file = "acute_mi_prevent_ingredients_marker_fold_baseline_risk.rds",
    nmr_file = "acute_mi_prevent_ingredients_marker_nmr_fold_baseline_risk.rds"
  ),
  chd = list(
    title = "Coronary Heart Disease", short = "CHD", y_max = 0.30,
    soc_file = "chd_revasc_prevent_ingredients_marker_fold_baseline_risk.rds",
    nmr_file = "chd_revasc_prevent_ingredients_marker_nmr_fold_baseline_risk.rds"
  ),
  tia_stroke = list(
    title = "TIA/Stroke", short = "TIA/Stroke", y_max = 0.20,
    soc_file = "tia_stroke_prevent_ingredients_marker_fold_baseline_risk.rds",
    nmr_file = "tia_stroke_prevent_ingredients_marker_nmr_fold_baseline_risk.rds"
  ),
  total_cvd = list(
    title = "Total CVD", short = "Total CVD", y_max = 0.50,
    soc_file = "total_cvd_prevent_ingredients_marker_fold_baseline_risk.rds",
    nmr_file = "total_cvd_prevent_ingredients_marker_nmr_fold_baseline_risk.rds"
  ),
  afib = list(
    title = "Atrial Fibrillation", short = "Afib", y_max = 0.31,
    soc_file = "afib_chargeaf_ingredients_marker_fold_baseline_risk.rds",
    nmr_file = "afib_chargeaf_ingredients_marker_nmr_fold_baseline_risk.rds"
  ),
  copd = list(
    title = "COPD", short = "COPD", y_max = 0.15,
    soc_file = "copd_smoking_ldl_marker_fold_baseline_risk.rds",
    nmr_file = "copd_smoking_ldl_marker_nmr_fold_baseline_risk.rds"
  )
)

primary_ids <- setdiff(names(disease_specs), "copd")

load_model_pair <- function(spec) {
  soc_path <- file.path(risk_dir, spec$soc_file)
  nmr_path <- file.path(risk_dir, spec$nmr_file)
  missing <- c(soc_path, nmr_path)[!file.exists(c(soc_path, nmr_path))]
  if (length(missing) > 0) {
    stop("Missing absolute-risk RDS file(s):\n", paste(missing, collapse = "\n"))
  }
  list(SOC = readRDS(soc_path), `SOC+NMR` = readRDS(nmr_path))
}

oof_store <- lapply(disease_specs, load_model_pair)


In [ ]:
# compute percentile incidence for each disease and model
percentile_rows <- lapply(names(disease_specs), function(id) {
  spec <- disease_specs[[id]]
  dplyr::bind_rows(lapply(names(oof_store[[id]]), function(model) {
    df <- compute_percentile_incidence_from_oof_risk(
      oof_store[[id]][[model]],
      t_star = t_star,
      risk_col = risk_col,
      n_percentiles = n_percentiles
    )
    df$disease <- spec$title
    df$model <- model
    df
  }))
})
# bind rows of percentile incidence for each disease and model
percentile_df <- dplyr::bind_rows(percentile_rows)
attr(percentile_df, "t_star") <- t_star

# set y-axis limits for each disease
y_limits <- setNames(
  lapply(disease_specs[primary_ids], function(spec) spec$y_max),
  vapply(disease_specs[primary_ids], function(spec) spec$title, character(1))
)

# plot observed incidence by predicted risk for ckm outcomes
p_enrich <- plot_percentile_incidence_faceted(
  percentile_df[percentile_df$disease %in% names(y_limits), ],
  y_limits = y_limits,
  t_star = t_star
)
p_enrich
ggsave(file.path(fig_dir, "ENRICH.png"), p_enrich, width = 7, height = 7, dpi = 600)


# plot observed incidence by predicted risk for copd 
p_copd_percentiles <- plot_percentile_incidence(
  percentile_df[percentile_df$disease == "COPD", ],
  title = "COPD",
  t_star = t_star,
  y_limits = c(0, disease_specs$copd$y_max)
)


options(repr.plot.height = 4, repr.plot.width = 3.5)
p_copd_percentiles
ggsave(
  file.path(fig_dir, "COPD_percentiles.png"),
  p_copd_percentiles,
  width = 3.5,
  height = 4,
  dpi = 300
)


In [ ]:
disease_models <- setNames(
  lapply(names(disease_specs), function(id) oof_store[[id]]),
  vapply(disease_specs, function(spec) spec$short, character(1))
)

top_incidence <- top_risk_incidence_from_oof_risk(
  disease_models,
  t_star = t_star,
  risk_col = risk_col,
  top_props = top_props,
  reference_model = "SOC"
)

top_incidence %>%
  mutate(obs_incidence_pct = round(100 * obs_incidence, 2)) %>%
  select(disease, model, top_label, n_top, n_events_top_by_tstar, obs_incidence_pct) %>%
  arrange(disease, top_label, model)


In [ ]:
primary_levels <- vapply(disease_specs[primary_ids], function(spec) spec$short, character(1))
top_label_levels <- c("Top 1%", "Top 5%", "Top 10%")
soc_col <- "#E64B35"
nmr_col <- "#4DBBD5"

paired_incidence <- function(df, disease_levels, top_levels) {
  df %>%
    mutate(
      disease = factor(as.character(disease), levels = disease_levels),
      top_label = factor(as.character(top_label), levels = top_levels),
      obs_incidence_pct = 100 * obs_incidence
    ) %>%
    filter(!is.na(disease)) %>%
    select(disease, top_label, model, obs_incidence_pct) %>%
    pivot_wider(names_from = model, values_from = obs_incidence_pct)
}

paired_primary <- paired_incidence(
  top_incidence,
  disease_levels = rev(primary_levels),
  top_levels = top_label_levels
)

p_fig_2d <- ggplot(paired_primary, aes(y = disease)) +
  geom_segment(
    aes(x = SOC, xend = `SOC+NMR`, yend = disease),
    linewidth = 0.75,
    color = "grey68"
  ) +
  geom_point(aes(x = SOC, color = "SOC", shape = "SOC"), size = 3.1) +
  geom_point(aes(x = `SOC+NMR`, color = "SOC+NMR", shape = "SOC+NMR"), size = 3.1) +
  facet_grid(. ~ top_label, axes = "all_y", axis.labels = "all_y") +
  scale_x_continuous(
    limits = c(0, 82),
    breaks = seq(0, 80, 20),
    labels = function(x) paste0(x, "%")
  ) +
  scale_color_manual(values = c("SOC" = soc_col, "SOC+NMR" = nmr_col), breaks = c("SOC", "SOC+NMR")) +
  scale_shape_manual(values = c("SOC" = 16, "SOC+NMR" = 17), breaks = c("SOC", "SOC+NMR")) +
  labs(
    title = "Observed Incidence in Top Predicted Risk Group",
    x = "Observed 5-year incidence",
    y = NULL,
    color = NULL,
    shape = NULL
  ) +
  theme_bw(base_size = 12) +
  theme(
    plot.title = element_text(hjust = 0.5, face = "bold", size = 13),
    legend.position = "bottom",
    panel.grid.major.y = element_blank(),
    panel.grid.minor = element_blank(),
    axis.ticks.y = element_blank(),
    strip.background = element_blank(),
    strip.text = element_text(size = 12, face = "bold")
  )

options(repr.plot.height = 2.9, repr.plot.width = 9)
p_fig_2d
ggsave(file.path(fig_dir, "FIG_2D.png"), p_fig_2d, width = 9, height = 2.9, dpi = 600)

## Supplemental outcome: COPD


In [ ]:
paired_copd <- paired_incidence(
  top_incidence,
  disease_levels = "COPD",
  top_levels = rev(top_label_levels)
)

x_max <- max(c(paired_copd$SOC, paired_copd$`SOC+NMR`), na.rm = TRUE)
x_max <- max(10, ceiling(x_max / 5) * 5)

p_copd_incidence <- ggplot(paired_copd, aes(y = top_label)) +
  geom_segment(
    aes(x = SOC, xend = `SOC+NMR`, yend = top_label),
    linewidth = 0.8,
    color = "grey68"
  ) +
  geom_point(aes(x = SOC, color = "SOC", shape = "SOC"), size = 3.1) +
  geom_point(aes(x = `SOC+NMR`, color = "SOC+NMR", shape = "SOC+NMR"), size = 3.1) +
  scale_x_continuous(
    limits = c(0, x_max),
    breaks = seq(0, x_max, by = ifelse(x_max <= 20, 5, 10)),
    labels = function(x) paste0(x, "%")
  ) +
  scale_color_manual(values = c("SOC" = soc_col, "SOC+NMR" = nmr_col), breaks = c("SOC", "SOC+NMR")) +
  scale_shape_manual(values = c("SOC" = 16, "SOC+NMR" = 17), breaks = c("SOC", "SOC+NMR")) +
  labs(
    title = "COPD",
    x = "Observed 5-year incidence",
    y = NULL,
    color = NULL,
    shape = NULL
  ) +
  theme_bw(base_size = 12) +
  theme(
    plot.title = element_text(hjust = 0.5, face = "bold", size = 13),
    legend.position = "bottom",
    panel.grid.major.y = element_blank(),
    panel.grid.minor = element_blank(),
    axis.ticks.y = element_blank()
  )

options(repr.plot.height = 2.4, repr.plot.width = 3.5)
p_copd_incidence
ggsave(file.path(fig_dir, "COPD_incidence.png"), p_copd_incidence, width = 3.5, height = 2.4, dpi = 600)
